# Exploratory Data Analysis

This notebook is dedicated to univariate and bivariate EDA for the Indian Liver Patient Dataset (ILPD). All generated diagrams are saved under `reports/figures/eda/`.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import kurtosis, skew

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

def resolve_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

ROOT = resolve_repo_root(Path.cwd())
DATA_PATH = ROOT / "data" / "raw" / "ILPD.csv"
FIGURES_DIR = ROOT / "reports" / "figures" / "eda"
UNIVARIATE_DIR = FIGURES_DIR / "univariate"
BIVARIATE_DIR = FIGURES_DIR / "bivariate"

for directory in [FIGURES_DIR, UNIVARIATE_DIR, BIVARIATE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FIG_DPI = 150
print(f"Repository root: {ROOT}")
print(f"Data source: {DATA_PATH}")
print(f"EDA figures: {FIGURES_DIR}")


## Load and prepare the dataset

In [ ]:
COLUMNS = [
    "Age",
    "Gender",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Protiens",
    "Albumin",
    "Albumin_Globulin_Ratio",
    "Target",
]

df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS)
df["Albumin_Globulin_Ratio"] = df["Albumin_Globulin_Ratio"].fillna(df["Albumin_Globulin_Ratio"].median())
df["Target"] = df["Target"].map({1: "Liver Disease", 2: "No Liver Disease"})
df["Gender"] = df["Gender"].astype("category")

numeric_columns = [
    "Age",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Protiens",
    "Albumin",
    "Albumin_Globulin_Ratio",
]

df.head()


In [ ]:
def save_figure(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    plt.close()

summary = pd.DataFrame(
    {
        "missing_values": df.isna().sum(),
        "unique_values": df.nunique(),
    }
)
summary.loc[numeric_columns, "skewness"] = df[numeric_columns].apply(skew)
summary.loc[numeric_columns, "kurtosis"] = df[numeric_columns].apply(kurtosis)
summary


## Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x="Target", ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].tick_params(axis="x", rotation=15)

sns.countplot(data=df, x="Gender", ax=axes[1])
axes[1].set_title("Gender distribution")

save_figure(UNIVARIATE_DIR / "categorical_distributions.png")


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, column in zip(axes.flat, numeric_columns):
    sns.histplot(df[column], kde=True, ax=ax)
    ax.set_title(f"Distribution of {column}")

for ax in axes.flat[len(numeric_columns):]:
    ax.axis("off")

save_figure(UNIVARIATE_DIR / "numeric_histograms.png")


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, column in zip(axes.flat, numeric_columns):
    sns.boxplot(data=df, y=column, ax=ax)
    ax.set_title(f"Boxplot of {column}")

for ax in axes.flat[len(numeric_columns):]:
    ax.axis("off")

save_figure(UNIVARIATE_DIR / "numeric_boxplots.png")


In [ ]:
df[numeric_columns].describe().T


## Bivariate analysis

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Gender", hue="Target")
plt.title("Gender vs target")
save_figure(BIVARIATE_DIR / "gender_vs_target.png")


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, column in zip(axes.flat, numeric_columns):
    sns.boxplot(data=df, x="Target", y=column, ax=ax)
    ax.set_title(f"{column} vs Target")
    ax.tick_params(axis="x", rotation=15)

for ax in axes.flat[len(numeric_columns):]:
    ax.axis("off")

save_figure(BIVARIATE_DIR / "numeric_vs_target_boxplots.png")


In [ ]:
pairplot_columns = [
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Albumin",
    "Albumin_Globulin_Ratio",
    "Target",
]

pairplot_grid = sns.pairplot(
    df[pairplot_columns],
    hue="Target",
    diag_kind="hist",
    corner=True,
)
pairplot_grid.figure.suptitle("Selected feature relationships", y=1.02)
pairplot_grid.figure.savefig(BIVARIATE_DIR / "selected_pairplot.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()
plt.close("all")


In [ ]:
corr_df = df.copy()
corr_df["Gender_Code"] = corr_df["Gender"].cat.codes
corr_df["Target_Code"] = corr_df["Target"].map({"No Liver Disease": 0, "Liver Disease": 1})

correlation_matrix = corr_df[numeric_columns + ["Gender_Code", "Target_Code"]].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation heatmap")
save_figure(BIVARIATE_DIR / "correlation_heatmap.png")


## Output summary

When you run this notebook, the generated figure set is written to:

- `reports/figures/eda/univariate/`
- `reports/figures/eda/bivariate/`

## Export notebook to HTML report

In [ ]:
import subprocess

HTML_DIR = ROOT / "produced_reports" / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = ROOT / "scripts" / "eda_univariate_bivariate.ipynb"

subprocess.run([
    "jupyter", "nbconvert",
    "--to", "html",
    str(NOTEBOOK_PATH),
    "--output-dir", str(HTML_DIR),
], check=True)

print(f"Exported HTML to: {HTML_DIR / NOTEBOOK_PATH.with_suffix('.html').name}")